# Inhoudsopgave
## Documentatie / uitleg
- Productoverzicht
- Stakeholder analyse
- Datavereisten
- Modelvereisten
- Onderhoud en hertraining
- Data pipeline
- Modellering
- Deployment
- CI/CD
- Monitoring
## Technisch onderdeel
- Data loading
- Data preprocessing and feature engineering
- Model training
- Deployment
- Monitoring

# Documentatie & uitleg

Hieronder wordt uitleg gegeven over de inhoud van het notebook.
### Productoverzicht:
Dit project maakt een intelligent retail analytics systeem voor BrightMart, een middelgrote retailer. Het doel van het systeem is om winkels efficiënter te maken door edge-based klantdetectie te combineren met cloudgebaseerde vraagvoorspellingen.

Het systeem bestaat uit twee modellen:
- Een cloud based model dat de vraag naar producten voorspelt op basis van historische verkoopgegevens.
- Een edge model dat het aantal klanten in een winkel inschat op basis van camerabeelden

Met dit systeem kunnen er real-time inzichten verkregen worden voor store managers en kan de voorraad beter bijgehouden worden voor het supply chain team.

### Stakeholders:
- Store managers: Hebben behoefte aan real time inzicht van de winkel bezetting en de omzet van een winkel.
- Supply chain team: Hebben behoefte aan een nauwkeurige voorspelling van verkoop.
- IT-afdeling: Wilt een data pipeline waar niet veel aan gedaan hoeft te worden.

### Datavereisten:
Het systeem maakt gebruik van twee soorten data:
- Retaildata: Deze data bevat informatie over winkel, product, datum en verkoop.
- Beelddata: Deze wordt gebruikt om klanten te detecteren en te tellen.

De eisen van de data zijn als volgt:
- Kwaliteit: De data moet schoon zijn. (Missende waardes worden verwijderd)
- Volume: De pipeline is schaalbaar en kan volume aan.
- Snelheid: Het edge model moet real-time voorspellingen kunnen maken.
- Privacy: De beelddata wordt lokaal verwerkt en niet opgeslagen.
- Veiligheid: Data wordt veilig opgeslagen en verwerkt binnen een beveiligd platform
- Vorm: De beelddata wordt in .npy bestanden aangeleverd.

### Model vereisten
Er worden twee modellen gebruikt:

#### Cloud model:
- Taak: Voorspellen van het aan verkochte items van aankomende dagen zodat het supply chain team spullen kan inkopen.
- Type model: RandomForestRegressor (SparkML)
- Eisen:
  - Zo laag mogelijke RMSE
  - Schaalbaar
  - MLflow integratie voor tracking en versiebeheer

#### Edge model:
- Taak: Het aantal klanten in een winkel voorspellen aan de hand van camerabeelden.
- Type model: Regressie (Lineaire regressie of RandomForest)
- Eisen:
  - Real time voorspellingen kunnen maken
  - Zo laag mogelijke RMSE

#### Onderhoud & hertraining:
Het systeem wordt ontworpen om zich aan te passen aan verandering.
- Modelprestaties worden gemonitord aan de hand van de RMSE.
- Data drift wordt gedetecteerd, bij data drift krijgt gebruiker een melding om hertraining in te plannen.
- Data drift detectie kan per model worden aangepast naar andere hoeveelheid.

#### Data pipeline:
De data pipeline bestaat uit:
- Data ingestion: Het inladen van de retail- en beelddata.
- Data cleaning: Het verwijderen van ongeldige en dubbele waardes
- Feature engineering: Het toevoegen van features zoals datum componenten en lag-variabelen.
- Data splitsen: Het maken van train/test sets.

#### Modellering:
De modelleringspipeline bevat:
- Feature engineering (VectorAssembler voor SparkML)
- Model training (RandomForest en Lineaire regressie)
- Evaluatie met RMSE
- Experiment tracking met behulp van MLFlow

#### Deployment:
De modellen worden geladen vanuit MLFlow en gebruikt voor voorspellingen.

#### CI/CD:
Het systeem ondersteund het volgende:
- Continuous Integration: Modellen worden bijgehouden in MLFlow.
- Continuous Deployment: Nieuwe modellen kunnen toegepast worden.

#### Monitoring:
Modelprestaties worden gemonitord:
- RMSE voor evaluatie.
- Drift detection vergelijkt voorspellingen met echte waardes.
- Bij afwijkingen krijgt gebruiker een melding om modellen opnieuw te trainen.

# Technisch onderdeel

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor as SparkRF
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.window import Window
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor as SKLearnRF

from PIL import Image
import mlflow
import os
import numpy as np
import pandas as pd
from abc import ABC, abstractmethod

# Data loading

In [0]:
store_demand_schema = StructType([
  StructField('date', DateType(), False),
  StructField('store', IntegerType(), False),
  StructField('item', IntegerType(), False),
  StructField('sales', IntegerType(), False)
])
ecom_schema = StructType([
  StructField('InvoiceNo', IntegerType(), False),
  StructField('StockCode', IntegerType(), False),
  StructField('Description', StringType(), False),
  StructField('Quantity', IntegerType(), False),
  StructField('InvoiceDate', StringType(), False),
  StructField('UnitPrice', FloatType(), False),
  StructField('CustomerID', IntegerType(), False),
  StructField('Country', StringType(), False)
])

base_path = "/Volumes/workspace/default/course_files/"

ecom_df = spark.read.csv(
    base_path + "ecom/data.csv",
    header=True,
    schema=ecom_schema
)

retail_train = spark.read.csv(
    base_path + "retail/train.csv",
    header=True,
    schema=store_demand_schema
)

retail_test = spark.read.csv(
    base_path + "retail/test.csv",
    header=True,
    schema=store_demand_schema
)

images_df = spark.read.format("binaryFile").load(
    base_path + "person_detection_small/images/"
).select("path")

images = np.load(base_path + "surv_camera/images_400.npy")
labels = np.load(base_path + "surv_camera/labels_400.npy")

print("Ecom rows:", ecom_df.count())
print("Retail train rows:", retail_train.count())

print("Images numpy shape:", images.shape)
print("Labels numpy shape:", labels.shape)

display(ecom_df)
display(retail_train)
display(images_df.limit(5))

# Data preprocessing and feature engineering

Hieronder worden functies aangemaakt om de data op te schonen en extra features aan te maken. Dit wordt gedaan middels zelf-gedefineerde functies zodat er modulair gewerkt kan worden en het makkelijk is om dingen aan te passen of te veranderen. 

Daarna worden de dataframes bewerkt door middel van de functies en wordt het dataframe klaargemaakt om gebruikt te worden in een machine-learning model.

In [0]:
def clean_ecom(df):
    return (
        df
        .filter(col("Quantity") > 0)
        .filter(col("UnitPrice") > 0) #Bij validatie kwam naar voren dat er ~2500 unitsprices waren die minder dan 0 UnitPrice waren.
        .filter(col("StockCode").isNotNull())
        .dropDuplicates()
    )

def clean_retail(df):
    return df \
        .filter(col("sales") > 0) \
        .dropDuplicates() \
        .dropna()

def create_retail_features(df):
    return (
        df
        .withColumn("year", year("date"))
        .withColumn("month", month("date"))
        .withColumn("day", dayofmonth("date"))
        .withColumn("full_date", col("date"))
    )

def create_ecom_features(df):
    return (
        df
        .withColumn(
            "invoice_timestamp",
            to_timestamp(
                "InvoiceDate",
                "M/d/yyyy H:mm"
            )
        )
        .withColumn(
            "revenue",
            col("Quantity") * col("UnitPrice")
        )
        .withColumn(
            "invoice_date",
            to_date(col("invoice_timestamp"))
        )
        .withColumn(
            "year",
            year("invoice_timestamp")
        )
        .withColumn(
            "month",
            month("invoice_timestamp")
        )
        .withColumn(
            "day",
            dayofmonth("invoice_timestamp")
        )
        .withColumn(
            "day_of_week",
            dayofweek("invoice_timestamp")
        )
    )


def create_retail_gold(df):
    return df.groupBy("store", "item", "year", "month", "day").agg(
    sum("sales").alias("sales")
    )

In [0]:
from abc import ABC, abstractmethod

def validate_retail_df(df):

    print(f"Rows: {df.count()}")

    print("\nSchema:")
    df.printSchema()

    # Null checks
    assert df.filter(col("date").isNull()).count() == 0, \
        "Null values found in date"

    assert df.filter(col("store").isNull()).count() == 0, \
        "Null values found in store"

    assert df.filter(col("item").isNull()).count() == 0, \
        "Null values found in item"

    assert df.filter(col("sales").isNull()).count() == 0, \
        "Null values found in sales"

    # Business rules
    assert df.filter(col("sales") < 0).count() == 0, \
        "Negative sales detected"

    assert df.filter(
        (col("month") < 1) | (col("month") > 12)
    ).count() == 0, \
        "Invalid month detected"

    assert df.filter(
        (col("day") < 1) | (col("day") > 31)
    ).count() == 0, \
        "Invalid day detected"

    print("\nRetail validation passed")


def validate_ecom_df(df):

    assert df.filter(
        col("InvoiceDate").isNull()
    ).count() == 0, \
        "Null InvoiceDate values found"

    assert df.filter(
        col("Quantity") <= 0
    ).count() == 0, \
        "Invalid Quantity values found"

    assert df.filter(
        col("UnitPrice") <= 0
    ).count() == 0, \
        "Invalid UnitPrice values found"

    print("E-commerce validation passed")

class MLPipeline(ABC):

    def apply(self, df):

        df_clean = self._clean(df)

        df_features = self._features(df_clean)

        self._validate(df_features)

        return df_features

    @abstractmethod
    def _clean(self, df):
        pass

    @abstractmethod
    def _features(self, df):
        pass

    @abstractmethod
    def _validate(self, df):
        pass

class RetailPipeline(MLPipeline):

    def _clean(self, df):
        return clean_retail(df)

    def _features(self, df):
        return create_retail_features(df)

    def _validate(self, df):
        validate_retail_df(df)

class EcomPipeline(MLPipeline):

    def _clean(self, df):
        return clean_ecom(df)

    def _features(self, df):
        return create_ecom_features(df)

    def _validate(self, df):
        validate_ecom_df(df)

In [0]:
retail_pipeline = RetailPipeline()
retail_features = retail_pipeline.apply(retail_train)

ecom_pipeline = EcomPipeline()
ecom_features = ecom_pipeline.apply(ecom_df)

retail_gold = create_retail_gold(retail_features)

# Lag-features toevoegen voor forecasting
window = Window.partitionBy(
    "store",
    "item"
).orderBy(
    "year",
    "month",
    "day"
)

retail_gold = (
    retail_gold
    .withColumn(
        "lag_1",
        lag("sales", 1).over(window)
    )
    .withColumn(
        "lag_7",
        lag("sales", 7).over(window)
    )
    .dropna()
)

ecom_gold = (
    ecom_features
    .groupBy(
        "year",
        "month"
    )
    .agg(
        sum("revenue").alias("monthly_revenue")
    )
    .orderBy(
        "year",
        "month"
    )
)

In [0]:
ecom_df.filter(col("UnitPrice") <= 0).count()

# Schaalbaarheid

Om de schaalbaarheid aan te tonen gaan we de dataset vergroten en door de pipeline heen halen. Er is te zien dat het langer duurt naarmate de dataset wordt vergroot maar dat dit geen extreem grote stappen zijn en gelijkmatig toe neemt.

In [0]:
import time

results = []

base_df = [retail_train, ecom_df]
for df in base_df:
    for multiplier in [1, 2, 4, 8]:

        test_df = df

        current = 1

        while current < multiplier:
            test_df = test_df.unionByName(test_df)
            current *= 2

        rows = test_df.count()

        start = time.time()

        if df == ecom_df:
            features = ecom_pipeline.apply(test_df)
        else:   
            features = retail_pipeline.apply(test_df)

        features.count()

        runtime = time.time() - start

        results.append(
            (multiplier, rows, runtime)
        )

for multiplier, rows, runtime in results:

    print(
        f"Multiplier={multiplier} | "
        f"Rows={rows:,} | "
        f"Runtime={runtime:.2f}s | "
    )

# Model training

## Cloud model training
Hieronder wordt er een cloud model gemaakt om een voorspelling te maken van het aantal "items" wat verkocht gaat worden. Dit wordt aan de hand van een RandomForestRegressor model gedaan omdat deze een aantal voordelen heeft:
  - Robuust tegen ruis
  - Kan niet lineaire verbanden ontdekken
  - Heeft geen ingewikkelde pre-processing nodig\
Hierdoor is RandomForestRegressor een goed baseline model.

Er wordt een assembler gebruikt die alle features omzet naar één vector omdat spark modellen zo werken.

## Retail data

In [0]:
# Assembler aanmaken
assembler = VectorAssembler(
    inputCols=["store", "item", "year", "month", "day", "lag_1", "lag_7"],
    outputCol="features"
)

# RandomForest model aanmaken
rf = SparkRF(
    featuresCol="features",
    labelCol="sales"
)
# Pipeline bouwen
pipeline = Pipeline(stages=[assembler, rf])

In [0]:
#Laatste 20% van de tijd gebruiken als testset
split_date = "2017-01-01"

train = retail_gold.filter(
    col("full_date") < split_date
)

test = retail_gold.filter(
    col("full_date") >= split_date
)

print("Train rows:", train.count())
print("Test rows:", test.count())

In [0]:
print("Training period:")
train.selectExpr(
    "min(full_date)",
    "max(full_date)"
).show()

print("Testing period:")
test.selectExpr(
    "min(full_date)",
    "max(full_date)"
).show()

# Evaluator aanmaken met RMSE als metric
evaluator = RegressionEvaluator(
    labelCol="sales",
    predictionCol="prediction",
    metricName="rmse"
)

# Tijdelijke opslaglocatie voor modellen aanmaken
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/course_files/tmp/"

paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [5, 10])
    .addGrid(rf.numTrees, [20, 50])
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=3
)

with mlflow.start_run():

    cv_model = cv.fit(train)

    preds = cv_model.transform(test)

    rmse = evaluator.evaluate(preds)

    best_rf = cv_model.bestModel.stages[-1]

In [0]:
mlflow.log_metric("rmse", rmse)

mlflow.log_param(
        "best_maxDepth",
        best_rf.getMaxDepth()
    )

mlflow.log_param(
        "best_numTrees",
        best_rf.getNumTrees
    )

mlflow.spark.log_model(
        cv_model.bestModel,
        "best_model"
    )

print(f"Best RMSE: {rmse}")

print(
        f"Best depth: {best_rf.getMaxDepth()}"
    )

print(
        f"Best trees: {best_rf.getNumTrees}"
    )

## E-com data

In [0]:
# Time-based split
split_date = "2011-10-01"

ecom_train = ecom_features.filter(
    col("invoice_date") < split_date
)

ecom_test = ecom_features.filter(
    col("invoice_date") >= split_date
)

ecom_assembler = VectorAssembler(
    inputCols=[
        "Quantity",
        "UnitPrice",
        "year",
        "month",
        "day",
        "day_of_week"
    ],
    outputCol="features"
)

ecom_rf = SparkRF(
    featuresCol="features",
    labelCol="revenue"
)

ecom_pipeline_ml = Pipeline(
    stages=[
        ecom_assembler,
        ecom_rf
    ]
)

print("Train rows:", ecom_train.count())
print("Test rows:", ecom_test.count())

In [0]:
ecom_evaluator = RegressionEvaluator(
    labelCol="revenue",
    predictionCol="prediction",
    metricName="rmse"
)

paramGrid = (
    ParamGridBuilder()
    .addGrid(ecom_rf.maxDepth, [5, 10])
    .addGrid(ecom_rf.numTrees, [20, 50])
    .build()
)

ecom_cv = CrossValidator(
    estimator=ecom_pipeline_ml,
    estimatorParamMaps=paramGrid,
    evaluator=ecom_evaluator,
    numFolds=3
)

In [0]:
with mlflow.start_run(run_name="ecommerce_model"):
    os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"
    os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/course_files/tmp/"

    ecom_cv_model = ecom_cv.fit(ecom_train)

    ecom_preds = ecom_cv_model.transform(ecom_test)

    ecom_rmse = ecom_evaluator.evaluate(ecom_preds)

    best_rf = ecom_cv_model.bestModel.stages[-1]

    mlflow.log_metric(
        "rmse",
        ecom_rmse
    )

    mlflow.log_param(
        "best_maxDepth",
        best_rf.getMaxDepth()
    )

    mlflow.log_param(
        "best_numTrees",
        best_rf.getNumTrees
    )

    mlflow.spark.log_model(
        ecom_cv_model.bestModel,
        "best_ecommerce_model"
    )

    print(f"RMSE: {ecom_rmse}")
    print(f"Best depth: {best_rf.getMaxDepth()}")
    print(f"Best trees: {best_rf.getNumTrees}")

### Experiment log voor cloud model
Hieronder worden de resultaten van de verschillende modellen getoond, er is duidelijk te zien dat de lag functies de prestaties hebben verbeterd. Index 4-5-6 zijn de modellen die getraind zijn zonder lag features en index 1-2-3 zijn de modellen die wel zijn getraind met lag features. Binnen het nieuwe model maakt het minder uit hoe uitgebreid de maxDepth is, daarom kan er een afweging gemaakt worden over wat belangrijker is. Snelheid van trainen, accuraatheid of generalisatie. Hier wordt later meer over verteld.

In [0]:
log = mlflow.search_runs()
cols = [
    c for c in log.columns
    if "param" in c.lower()
       or "metric" in c.lower()
]
print(cols)
display(log)

## Edge model training
Nu er een cloudmodel is getraind om de verkoop te voorspellen, wordt een edge model ontwikkeld dat het aantal klanten in de winkel inschat. Dit model maakt gebruik van camerabeelden en wordt lokaal uitgevoerd op edge-apparaten voor real-time inzichten.

Voor het edge model worden twee regressiemodellen vergeleken: lineaire regressie en een Random Forest model. Aangezien de dataset beschikbaar is als NumPy-arrays (.npy), worden de beelden direct als numerieke input gebruikt, zonder een volledige computer vision pipeline te implementeren.

In plaats van complexe deep learning-modellen is bewust gekozen voor lichte modellen. Deze keuze is gemaakt om te voldoen aan de beperkingen van edge-apparaten, zoals beperkte rekenkracht en geheugen.

Om beide modellen te trainen worden er twee datasets gemaakt, een kleine variant waar de images verkleind worden omdat RandomForest er te lang over doet.  

In [0]:
# Images dataset verkleinen naar 64x64
images_small = np.array([
    np.array(Image.fromarray(img).resize((64, 64)))
    for img in images
])
# Labels defineren (aantal mensen in de foto)
y = labels

# Images hervormen zodat ze als input gebruikt kunnen worden. 
X_small = images_small.reshape(len(images_small), -1)

# Train test split maken
X_train_small, X_test_small, y_train_small, y_test_small = train_test_split(
    X_small, y, test_size=0.2, random_state=42
)

# Images hervormen zodat ze als input gebruikt kunnen worden
X = images.reshape(len(images), -1)

# Train test split maken
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [0]:
# (Tijdelijke) opslagplek voor de modellen defineren
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/default/course_files/tmp/"

# Lineaire regressie
with mlflow.start_run(run_name="LinearRegression"):
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    
    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(model, "edge_model_linear")
    
    print("Linear RMSE:", rmse)

# RandomForestRegressor
with mlflow.start_run(run_name="RandomForest"):
    
    model = SKLearnRF(n_estimators=50)
    model.fit(X_train_small, y_train_small)
    
    preds = model.predict(X_test_small)
    mse = mean_squared_error(y_test_small, preds)
    rmse = np.sqrt(mse)
    
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.sklearn.log_model(model, "edge_model_rf")
    
    print("RF RMSE:", rmse)

### Experiment log edge model
Hieronder is te zien dat een linaire regressie model iets beter presteert, ook is deze sneller getraind. Daarom is het aangeraden om dit model te gebruiken als edge model. Een lineaire regressie model is licht, simpel en makkelijk te interpreteren.

In [0]:
edge_log = log[["run_id", "experiment_id","metrics.rmse","params.model_type","params.n_estimators"]]
edge_log = edge_log.dropna(subset=["params.model_type"])
display(edge_log)

# Deployment
Nu de data door de pipeline heen is en alle modellen getraind zijn, is het tijd voor de deployment. Om dit te doen zijn er run id's nodig die in de log staan, deze zijn te vinden in DataBricks onder "AI/ML" -> "Experiments". Om te weten welke het beste presteert kunnen we de bovenstaande dataframes raadplegen waar de benodigde informatie in staat.

### Cloud model deployment
Voor het cloud model wordt het het model gebruikt wat getraind is op de dataset waar lag features zijn toegevoegd en waar de maxDepth 5 is. Het model presteert bijna even goed als die van 10/15 alleen is een maxDepth van 5 sneller dan de hogere maxDepths. Door is experiments te kijken is dit ook terug te zien:
- maxDepth 5: 23 seconden
- maxDepth 10: 30 seconden
- maxDepth 15: 96 seconden\
Dit lijkt in eerste instantie niet veel maar wanneer er veel meer data binnen komt, schelen deze aantallen enorm. 

Er zou eventueel voor de maxDepth van 10 gekozen kunnen worden als de precisie van het model belangrijker is dan de tijd die gebruikt wordt voor het trainen.

In [0]:
# Model ophalen
model_uri = "runs:/ce70553bf9494add844e4050b417480a/model_depth_5"
# Model laden met bovenstaand variabel
loaded_model = mlflow.spark.load_model(model_uri)
#Model laten voorspellen
preds_cloud = loaded_model.transform(test)
#Voorspelling laten zien
display(preds_cloud.select("sales", "prediction"))

### Edge model deployment
Voor het edge model zijn er twee modellen getraind en is er overduidelijk een beter, zowel in snelheid als in prestatie. Daarom wordt hieronder het linair regressiemodel aangeroepen op dezelfde manier zoals dat is gebeurt bij het cloud model.

In [0]:
# Model ophalen
model_uri = "runs:/8fecfb4e426d4cd7b3f2926fd8062652/edge_model_linear"
# Model laden
edge_model_loaded = mlflow.sklearn.load_model(model_uri)
# Voorspellingen maken
sample = X_test[:5]
actual = y_test[:5].ravel() # Naar 1D veranderen

preds_edge = edge_model_loaded.predict(sample)
preds_edge = preds_edge.ravel() # Naar 1D veranderen

compare_df = pd.DataFrame({
    "actual": actual,
    "predicted": preds_edge
})
compare_df

# Model monitoring
Als laatste gaan we een belangrijk stuk toevoegen, namelijk het monitoren van de modellen en binnenstromende data. Het kan zo zijn dat de data veranderd en/of dat de modellen niet meer toereikend zijn. Het is belangrijk om zo snel mogelijk in te kunnen grijpen als dit gebeurt.

### Performance monitoring:

In [0]:
# Cloud monitoring
# (Nieuwe) predictions maken
preds_cloud_performance = loaded_model.transform(test)
# RMSE berekenen met evaluator van eerder
rmse = evaluator.evaluate(preds_cloud_performance)

print("Cloud RMSE:", rmse)

In [0]:
# Edge monitoring
# (Nieuwe) predictions maken
preds_edge_performance = edge_model_loaded.predict(X_test)
# RMSE berekenen
rmse = np.sqrt(mean_squared_error(y_test, preds_edge_performance))

print("Edge RMSE:", rmse)

### Drift detection:
In deze drift detection worden de gemiddeldes van de predictions en echte data vergeleken. Er is onderscheid gemaakt tussen de verschillende modellen omdat deze waardes gemiddeld ook verder uit elkaar liggen. Als dezelfde waardes gehanteerd zouden worden, zou er al sprake kunnen zijn van drift bij het edge model zonder dat dit wordt opgemerkt. Uiteraard kan de threshhold veranderd worden om eerder/later drift te detecteren.

Als er drift wordt gedetecteerd dan wordt er een nieuwe model getraind.

In [0]:
def retrain_cloud(train_df, assembler):
    from pyspark.ml.regression import RandomForestRegressor
    from pyspark.ml import Pipeline
    import mlflow

    rf = RandomForestRegressor(
        maxDepth=5,
        featuresCol="features",
        labelCol="sales"
    )

    pipeline = Pipeline(stages=[assembler, rf])

    model = pipeline.fit(train_df)

    print("Cloud model retrained")

    mlflow.spark.log_model(model, "cloud_model_retrained")

    return model

def retrain_edge(X_train, y_train):
    from sklearn.linear_model import LinearRegression
    import mlflow

    model = LinearRegression()
    model.fit(X_train, y_train)

    print("Edge model (Linear Regression) retrained")

    # log opnieuw in MLflow
    mlflow.sklearn.log_model(model, "edge_model_linear_retrained")

    return model

def detect_drift(preds, y_test, model=None):
    import numpy as np

    preds = np.array(preds)
    actual = np.array(y_test)

    avg_pred = np.mean(preds)
    avg_actual = np.mean(actual)

    print("Pred avg:", avg_pred)
    print("Actual avg:", avg_actual)

    if model == "edge":
        threshold = 5
    elif model == "cloud":
        threshold = 10
    else:
        raise ValueError("Model must be 'edge' or 'cloud'")

    if np.abs(avg_pred - avg_actual) > threshold:
        print("Drift detected, retraining")

        if model_type == "edge":
            return retrain_edge(X_train, y_train)

        elif model_type == "cloud":
            return retrain_cloud(train, assembler)

    else:
        print("No drift detected")



cdt = preds_cloud.select("prediction", "sales").toPandas()
cloud_drift = detect_drift(cdt["prediction"].values, cdt["sales"].values, "cloud")
edge_drift = detect_drift(preds_edge, y_test, "edge")
